# Preprocess GMFD from glade

In [1]:
import xarray as xr
import xagg as xa
import pandas as pd
import numpy as np
import os
import glob
import re
from matplotlib import pyplot as plt
from cartopy import crs as ccrs
import cmocean
from tqdm.notebook import tqdm

from distributed import Client


from funcs_support import get_params,get_filepaths,utility_save
from funcs_aux import _remove_chunk_encoding
dir_list = get_params()

In [ ]:
# Start dask client
client = Client()
display(client)

In [3]:
# File format: 
# /glade/campaign/collections/rda/data/d314000/0.25deg/3hrly/1951/tas_0p25_3hourly_1951-1951.nc
fn_base = '/glade/campaign/collections/rda/data/d314000/0.25deg/3hrly/'

# Variable to load
#var = 'tas'; var_out = 'tas'
var = 'tasmax'; var_out = 'tasmax'

params_subset = {'lat':slice(-57,87),'lon':slice(-180,180)}

# MUST SET SCRATCH DIRECTORY HERE
output_dir = dir_list['raw']+'GMFD/'

In [ ]:
for yr in tqdm(np.arange(1981,2011)):
    # Get filename
    fn_open = fn_base+str(yr)+'/'+var+'_0p25_3hourly_'+str(yr)+'-'+str(yr)+'.nc'

    timestr = str(yr)+'0101-'+str(yr)+'1231'
    output_fn = output_dir+var_out+'_day_GMFD_historical_reanalysis_'+timestr+'.zarr'

    file_exists = os.path.exists(output_fn)

    if not file_exists:
        # Open file
        ds = xr.open_dataset(fn_open)
        ds = ds.chunk({'lat':100,'lon':100,'time':-1})
        # Standardize in name standards
        ds = xa.fix_ds(ds)
        # Subset geographically
        ds = ds.sel(**params_subset)
        
        # Get mean daily temperature
        ds = ds.resample({'time':'1D'}).mean()

        if var_out != var:
            ds = ds.rename({var:var_out})
        
        ds.attrs['SOURCE'] = 'preprocess_GMFD.ipynb'
        ds.attrs['DESCRIPTION'] = 'GMFD, from glade, daily means from from 3hrly, standardized to file system standards'

        ds = ds.chunk({'lat':100,'lon':100,'time':-1})
        ds = _remove_chunk_encoding(ds)
        
        utility_save(ds.drop_encoding(),output_fn)
    else:
        print(output_fn+' exists, skipped!')

In [6]:
fns = glob.glob(output_dir+'/'+var_out+'_*.zarr')

try:
    ds_out = xr.open_mfdataset(fns,
                          data_vars='minimal', coords='minimal', join='exact', compat='override')
except:
    ds_out = xr.concat([xr.open_dataset(fn,chunks='auto') for fn in np.sort(fns)],dim='time')
    ds_out = ds_out.sortby('time').drop_duplicates('time')    

ds_out = ds_out.chunk({'time':30})
# Since chunks are changed, have to remove the chunk encoding from the
# original files, otherwise saving gets confused
ds_out = _remove_chunk_encoding(ds_out)

timestr = (re.sub(r'\-','',str(ds_out.time.min().values)[0:10])+'-'+
           re.sub(r'\-','',str(ds_out.time.max().values)[0:10]))
fn_out = dir_list['raw']+'GMFD/'+var_out+'_day_GMFD_historical_reanalysis_'+timestr+'.zarr'

utility_save(ds_out,fn_out)

/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/zarr/api/asynchronous.py:227: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/climate_raw_derecho/GMFD/tas_day_GMFD_historical_reanalysis_19570101-19801231.zarr saved!
